In [0]:
from utils.logger import get_logger
from utils.ddl import build_column_ddl
from etl_config.gold_fact_config import FACT_CONFIG

logger = get_logger("gold_fact_setup")

In [0]:
for fact_name, cfg in FACT_CONFIG.items():
    full_table_name = cfg.target_table

    col_ddl = build_column_ddl(cfg.columns)

    spark.sql(
        f"""
            create table if not exists {full_table_name} (
                {col_ddl}
            )
            using delta
            comment :table_description
        """,
        args={"table_description": cfg.table_description},
    )
    logger.info(f"Fact table created/verified: {full_table_name}")

    spark.sql(
        f"""
            alter table {full_table_name}
            set tblproperties (
                'delta.autoOptimize.optimizeWrite' = 'true',
                'delta.autoOptimize.autoCompact'   = 'true',
                'description' = :table_description
            )
        """,
        args={"table_description": cfg.table_description},
    )

    for c in cfg.columns:
        if c.comment:
            try:
                spark.sql(
                    f"comment on column {full_table_name}.{c.name} is :comment",
                    args={"comment": c.comment}
                )
                logger.info(f"Added comment to column: {full_table_name}.{c.name}")
            except Exception as e:
                logger.warning(f"Could not add comment for {full_table_name}.{c.name}: {e}")

    logger.info(f"Metadata applied for: {full_table_name}")

logger.info("Gold fact setup complete.")

In [0]:
%sql
use acme_catalog.gold;

select * from information_schema.tables where table_schema = 'gold' order by table_name;

select * from information_schema.columns where table_schema = 'gold' and table_name = 'fact_orders' order by ordinal_position;